In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
data_path = "/home/bmb/haxx/working/ceisel_mumm/raws/"
data_ann_path = "/home/bmb/haxx/working/ceisel_mumm/"

In [ ]:
! ls {data_path}

In [ ]:
# !mkdir {data_path}/GSM8287434_RGC_24+48h_ranger
# !mv {data_path}/GSM8287434_RGC_24+48h_*.gz {data_path}/GSM8287434_RGC_24+48h_ranger

# !mkdir {data_path}/GSM8287438_RGC_12+72h_ranger
# !mv {data_path}/GSM8287438_RGC_12+72h_*.gz {data_path}/GSM8287438_RGC_12+72h_ranger

In [ ]:
annotations = pd.read_csv(data_ann_path + "ceisel_adata_metadata.csv")

In [ ]:
print(annotations.shape)
annotations.head()

In [ ]:
stripped_annotation_ids = [id.split("_")[2] for id in annotations['Cell.ID']]
stripped_annotation_ids
file_assignment = ["_".join(id.split("_")[:2]) for id in annotations['Cell.ID']]
annotations['Stripped.Cell.ID'] = stripped_annotation_ids
annotations['File.Assignment'] = file_assignment
annotations.set_index('Stripped.Cell.ID',inplace=True)

middle_annotations = annotations[annotations['File.Assignment'] == '24_48']
outer_annotations = annotations[annotations['File.Assignment'] == '12_72']
middle_annotations.head()
outer_annotations.head()


In [ ]:
# unique_annotation_ids = set(annotations['Cell.ID'])
# len(set(annotations['Cell.ID'])),annotations.shape

In [ ]:
middle_raw = sc.read_10x_mtx(
    "/home/bmb/haxx/working/ceisel_mumm/raws/GSM8287434_RGC_24+48h_ranger",
    prefix="GSM8287434_RGC_24+48h_"
)
print(middle_raw.shape)
outer_raw = sc.read_10x_mtx(
    "/home/bmb/haxx/working/ceisel_mumm/raws/GSM8287438_RGC_12+72h_ranger",
    prefix="GSM8287438_RGC_12+72h_"
)
print(outer_raw.shape)

In [ ]:
middle_ann_names = set(middle_annotations.index)
matching_middle = [id in middle_ann_names for id in middle_raw.obs_names]
print(f"middle: {np.sum(matching_middle)} out of {middle_raw.shape[0]}")
middle_raw.obs['annotated'] = matching_middle
outer_ann_names = set(outer_annotations.index)
matching_outer = [id in outer_ann_names for id in outer_raw.obs_names]
print(f"outer: {np.sum(matching_outer)} out of {outer_raw.shape[0]}")
outer_raw.obs['annotated'] = matching_outer


In [ ]:
middle_annotated = middle_raw[middle_raw.obs['annotated']]
outer_annotated = outer_raw[outer_raw.obs['annotated']]
middle_annotated.obs = middle_annotated.obs.join(middle_annotations)
outer_annotated.obs = outer_annotated.obs.join(outer_annotations)
middle_annotated.obs.head()
outer_annotated.obs.head()


In [ ]:
joint_raw = sc.anndata.concat([middle_annotated,outer_annotated])
joint_raw.shape

In [ ]:
sc.pp.calculate_qc_metrics(joint_raw, inplace=True)

In [ ]:
split_names = [n.split("-") for n in joint_raw.obs_names]
suffix = [n[-1] for n in split_names]

joint_raw.obs["library"] = suffix

plt.figure()
plt.hist(joint_raw.obs["total_counts"],bins=np.arange(0,10000,100))
plt.show()

plt.figure()
plt.hist(suffix)
plt.show()

plt.figure()
for suffix in set(suffix):
    masked = joint_raw[joint_raw.obs["library"] == suffix]
    plt.hist(masked.obs["total_counts"],bins=np.arange(0,4000,100),alpha=0.3,label=suffix)
plt.legend()
plt.show()

In [ ]:
sc.pp.filter_genes(joint_raw, min_cells=100)
sc.pp.filter_cells(joint_raw, min_counts=1000)

copy = joint_raw.copy()
sc.pp.log1p(copy)
sc.pp.highly_variable_genes(copy, n_top_genes=5000)

joint = joint_raw[:,copy.var["highly_variable"]].copy()
# sc.pp.downsample_counts(joint,counts_per_cell=1000)
sc.pp.normalize_total(joint,exclude_highly_expressed=True)
sc.pp.log1p(joint)

In [ ]:
print("Computing PCA")
sc.pp.pca(joint, n_comps=50)
print("Computing neighbors")
sc.pp.neighbors(joint)
print("Computing UMAP")
sc.tl.umap(joint)
sc.pl.umap(joint)


In [ ]:
log_total_counts = np.log1p(joint.obs['total_counts'])
joint.obs['log_total_counts'] = log_total_counts
sc.pl.umap(joint, color=["log_total_counts"])

sc.pl.umap(joint, color=["library"])

In [ ]:
# time = [c.split("_")[0].strip("h") for c in joint.obs['Condition']]
# condition = ["_".join(c.split("_")[1:]) for c in joint.obs['Condition']]
# set(condition)
# joint.obs['exp_condition'] = condition
# joint.obs['time'] = np.array(time).astype(int)
# joint.obs
# sc.pl.umap(joint, color=["time"])

# joint.obs['Condition']

In [ ]:
# fuzz = np.random.rand(joint.shape[0])*10
# plt.figure()
# plt.scatter(
#     joint.obs['time']+fuzz,
#     joint.obs['log_total_counts'],
#     s=1
# )
# plt.show()
plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [joint.obs['log_total_counts'][joint.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()



In [ ]:
# Load h5s
# rod_control = sc.read_10x_h5(data_path + "GSM8287430_RodD7ControlReplicate1.h5")


In [ ]:
# Annotation 
# - Timecourse 
# - Cell type (especially target (RGC/rods)) 
#      * microglia doublet warning here
# - Can we basically establish partial death, then regeneration 
# - peak death at 48h
# - regen starting at 72 (proliferation is starting)
# - We don't expect a developmental trajectory in general
# - PCA/ICA <- 

# - PCA delta between untreated treated
# - project onto apoptosis and necroptosis (and parthanatos if possible)

# - GSEA on raw deltas 

# Isolate Control Timecourse